- _exponent._bronze_allscripts_tw_works.dbo_order_activity_header
- _exponent._bronze_allscripts_tw_works.dbo_order_result_mapper
- _exponent._bronze_allscripts_tw_works.dbo_result_text
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item
- _exponent._bronze_allscripts_tw_works.dbo_vendor_item_extension

### Source Tables:
- _exponent._bronze_allscripts_tw_works_vw.dbo_item_result (measurement header - 413M records)
- _exponent._bronze_allscripts_tw_works_vw.dbo_result (measurement values/results - 427M records)
- _exponent._bronze_allscripts_tw_works_vw.dbo_item_finding (findings header - 283M records)
- _exponent._bronze_allscripts_tw_works_vw.dbo_finding (finding values - 298M records)

### To Do:
- Map QODE to OMOP measurement_concept_id using domain_source_to_concept
- Leverage LOINC codes from RIDLOINCCodeList for better concept mapping
- Map measurement_type_concept_id (default: 44818702 = Lab result)
- Link measurements to visits via ActivityHeaderID → order_activity_header → EncounterID
- Map unit_concept_id from UnitsDE/UnitsDET
- Parse reference ranges from ShortRefRange field
- Link to provider_id once provider table is populated

### Notes:
- PERSON and VISIT_OCCURRENCE must run before MEASUREMENT
- dbo_item_result.ID is the measurement identifier
- dbo_item_result.CurrentID links to dbo_result.ID for actual values
- dbo_item_result.PatientID links to dbo_person.ID
- Filters for NumericResult IS NOT NULL (quantitative data only)
- QODE = measurement type code
- LOINC codes available in RIDLOINCCodeList column
- Starting with dbo_item_result/dbo_result (can add findings later)

In [ ]:
%sql
-- Check measurement data availability (sample to avoid timeout)
SELECT 
    COUNT(*) as sample_measurements,
    COUNT(DISTINCT PatientID) as unique_patients
FROM (
    SELECT ir.PatientID 
    FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_item_result` ir
    LEFT JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_result` r
      ON ir.CurrentID = r.ID
    WHERE ir.PatientID IS NOT NULL 
      AND r.NumericResult IS NOT NULL
    LIMIT 10000
)

# Transformation

In [ ]:
source = 'allscripts_tw'

In [ ]:
silver_measurement_df = spark.sql(f'''
SELECT 
  source_to_person.person_id,
  COALESCE(meas_concept.omop_concept_id, 0) AS measurement_concept_id,  -- Map QODE or LOINC to OMOP concept
  CAST(COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS DATE) AS measurement_date,
  COALESCE(r.ClinicalDTTM, ir.PerformedDTTM) AS measurement_datetime,
  NULL AS measurement_time,
  44818702 AS measurement_type_concept_id,  -- 44818702 = Lab result
  NULL AS operator_concept_id,  -- TODO: Parse from AbnormalFlagType if needed
  r.NumericResult AS value_as_number,
  NULL AS value_as_concept_id,  -- TODO: Map answer codes to concepts if categorical
  0 AS unit_concept_id,  -- TODO: Map UnitsDE/UnitsDET to OMOP unit concepts
  NULL AS range_low,  -- TODO: Parse from ShortRefRange field
  NULL AS range_high,  -- TODO: Parse from ShortRefRange field
  NULL AS provider_id,  -- TODO: Map WhoDidItID once provider table is populated
  source_to_visitor_occurrence.visit_occurrence_id,
  NULL AS visit_detail_id,
  CONCAT('{source}', ' | ', ir.ID) AS measurement_source_value,
  COALESCE(meas_concept.source_concept_id, 0) AS measurement_source_concept_id,
  r.UnitsDET AS unit_source_value,
  0 AS unit_source_concept_id,
  CAST(r.NumericResult AS STRING) AS value_source_value,
  NULL AS measurement_event_id,
  NULL AS meas_event_field_concept_id,
  '{source}' AS source_system
FROM `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_item_result` ir
INNER JOIN _exponent.omop_mapping.source_to_person
  ON CONCAT('{source}', ' | ', ir.PatientID) = source_to_person.person_source_value
  AND source_to_person.active_flag = TRUE
INNER JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_result` r
  ON ir.CurrentID = r.ID
  AND r.NumericResult IS NOT NULL  -- CRITICAL: Only numeric measurements
LEFT JOIN `_exponent`.`_bronze_allscripts_tw_works_vw`.`dbo_order_activity_header` oah
  ON ir.ActivityHeaderID = oah.ID
LEFT JOIN _exponent.omop_mapping.source_to_visitor_occurrence
  ON CONCAT('{source}', ' | ', oah.EncounterID) = source_to_visitor_occurrence.visit_occurrence_source_value
  AND source_to_visitor_occurrence.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept meas_concept
  ON meas_concept.source_id = ir.QODE
  AND meas_concept.domain_id = 'Measurement'
  AND meas_concept.source_system = '{source}'
WHERE ir.ID IS NOT NULL
  AND ir.PatientID IS NOT NULL
''')

display(silver_measurement_df)
silver_measurement_df.createOrReplaceTempView("silver_measurement")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.measurement AS t
USING silver_measurement AS s
ON t.measurement_source_value = s.measurement_source_value

WHEN MATCHED AND (
     NOT (t.person_id <=> s.person_id)
  OR NOT (t.measurement_concept_id <=> s.measurement_concept_id)
  OR NOT (t.measurement_date <=> s.measurement_date)
  OR NOT (t.measurement_datetime <=> s.measurement_datetime)
  OR NOT (t.measurement_time <=> s.measurement_time)
  OR NOT (t.measurement_type_concept_id <=> s.measurement_type_concept_id)
  OR NOT (t.operator_concept_id <=> s.operator_concept_id)
  OR NOT (t.value_as_number <=> s.value_as_number)
  OR NOT (t.value_as_concept_id <=> s.value_as_concept_id)
  OR NOT (t.unit_concept_id <=> s.unit_concept_id)
  OR NOT (t.range_low <=> s.range_low)
  OR NOT (t.range_high <=> s.range_high)
  OR NOT (t.provider_id <=> s.provider_id)
  OR NOT (t.visit_occurrence_id <=> s.visit_occurrence_id)
  OR NOT (t.visit_detail_id <=> s.visit_detail_id)
  OR NOT (t.measurement_source_concept_id <=> s.measurement_source_concept_id)
  OR NOT (t.unit_source_value <=> s.unit_source_value)
  OR NOT (t.unit_source_concept_id <=> s.unit_source_concept_id)
  OR NOT (t.value_source_value <=> s.value_source_value)
  OR NOT (t.measurement_event_id <=> s.measurement_event_id)
  OR NOT (t.meas_event_field_concept_id <=> s.meas_event_field_concept_id)
  OR NOT (t.source_system <=> s.source_system)
)
THEN UPDATE SET
  t.person_id                      = s.person_id,
  t.measurement_concept_id         = s.measurement_concept_id,
  t.measurement_date               = s.measurement_date,
  t.measurement_datetime           = s.measurement_datetime,
  t.measurement_time               = s.measurement_time,
  t.measurement_type_concept_id    = s.measurement_type_concept_id,
  t.operator_concept_id            = s.operator_concept_id,
  t.value_as_number                = s.value_as_number,
  t.value_as_concept_id            = s.value_as_concept_id,
  t.unit_concept_id                = s.unit_concept_id,
  t.range_low                      = s.range_low,
  t.range_high                     = s.range_high,
  t.provider_id                    = s.provider_id,
  t.visit_occurrence_id            = s.visit_occurrence_id,
  t.visit_detail_id                = s.visit_detail_id,
  t.measurement_source_concept_id  = s.measurement_source_concept_id,
  t.unit_source_value              = s.unit_source_value,
  t.unit_source_concept_id         = s.unit_source_concept_id,
  t.value_source_value             = s.value_source_value,
  t.measurement_event_id           = s.measurement_event_id,
  t.meas_event_field_concept_id    = s.meas_event_field_concept_id,
  t.source_system                  = s.source_system

WHEN NOT MATCHED THEN INSERT (
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id,
  source_system
)
VALUES (
  s.person_id,
  s.measurement_concept_id,
  s.measurement_date,
  s.measurement_datetime,
  s.measurement_time,
  s.measurement_type_concept_id,
  s.operator_concept_id,
  s.value_as_number,
  s.value_as_concept_id,
  s.unit_concept_id,
  s.range_low,
  s.range_high,
  s.provider_id,
  s.visit_occurrence_id,
  s.visit_detail_id,
  s.measurement_source_value,
  s.measurement_source_concept_id,
  s.unit_source_value,
  s.unit_source_concept_id,
  s.value_source_value,
  s.measurement_event_id,
  s.meas_event_field_concept_id,
  s.source_system
);

In [ ]:
%sql
SELECT * FROM _exponent.omop_silver.measurement
LIMIT 10

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_measurement (
    source_system,
    measurement_source_value,
    person_id,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.measurement_source_value,
    s.person_id,
    TRUE AS active_flag,
    current_timestamp() AS created_tsp,
    current_timestamp() AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 
        source_system, 
        measurement_source_value,
        person_id
    FROM _exponent.omop_silver.measurement
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_measurement x
  ON s.measurement_source_value = x.measurement_source_value;

In [ ]:
%sql
SELECT * FROM _exponent.omop_mapping.source_to_measurement
LIMIT 20

In [ ]:
%sql
MERGE INTO _exponent.omop.measurement AS gold_meas
USING (
  SELECT
    source_to_measurement.measurement_id,
    s.person_id,
    s.measurement_concept_id,
    s.measurement_date,
    s.measurement_datetime,
    s.measurement_time,
    s.measurement_type_concept_id,
    s.operator_concept_id,
    s.value_as_number,
    s.value_as_concept_id,
    s.unit_concept_id,
    s.range_low,
    s.range_high,
    s.provider_id,
    s.visit_occurrence_id,
    s.visit_detail_id,
    s.measurement_source_value,
    s.measurement_source_concept_id,
    s.unit_source_value,
    s.unit_source_concept_id,
    s.value_source_value,
    s.measurement_event_id,
    s.meas_event_field_concept_id
  FROM _exponent.omop_silver.measurement s
  JOIN _exponent.omop_mapping.source_to_measurement
    ON source_to_measurement.measurement_source_value = s.measurement_source_value
   AND source_to_measurement.active_flag = TRUE
) AS src
ON gold_meas.measurement_id = src.measurement_id

WHEN MATCHED THEN UPDATE SET
  gold_meas.person_id                     = src.person_id,
  gold_meas.measurement_concept_id        = src.measurement_concept_id,
  gold_meas.measurement_date              = src.measurement_date,
  gold_meas.measurement_datetime          = src.measurement_datetime,
  gold_meas.measurement_time              = src.measurement_time,
  gold_meas.measurement_type_concept_id   = src.measurement_type_concept_id,
  gold_meas.operator_concept_id           = src.operator_concept_id,
  gold_meas.value_as_number               = src.value_as_number,
  gold_meas.value_as_concept_id           = src.value_as_concept_id,
  gold_meas.unit_concept_id               = src.unit_concept_id,
  gold_meas.range_low                     = src.range_low,
  gold_meas.range_high                    = src.range_high,
  gold_meas.provider_id                   = src.provider_id,
  gold_meas.visit_occurrence_id           = src.visit_occurrence_id,
  gold_meas.visit_detail_id               = src.visit_detail_id,
  gold_meas.measurement_source_value      = src.measurement_source_value,
  gold_meas.measurement_source_concept_id = src.measurement_source_concept_id,
  gold_meas.unit_source_value             = src.unit_source_value,
  gold_meas.unit_source_concept_id        = src.unit_source_concept_id,
  gold_meas.value_source_value            = src.value_source_value,
  gold_meas.measurement_event_id          = src.measurement_event_id,
  gold_meas.meas_event_field_concept_id   = src.meas_event_field_concept_id

WHEN NOT MATCHED THEN INSERT (
  measurement_id,
  person_id,
  measurement_concept_id,
  measurement_date,
  measurement_datetime,
  measurement_time,
  measurement_type_concept_id,
  operator_concept_id,
  value_as_number,
  value_as_concept_id,
  unit_concept_id,
  range_low,
  range_high,
  provider_id,
  visit_occurrence_id,
  visit_detail_id,
  measurement_source_value,
  measurement_source_concept_id,
  unit_source_value,
  unit_source_concept_id,
  value_source_value,
  measurement_event_id,
  meas_event_field_concept_id
)
VALUES (
  src.measurement_id,
  src.person_id,
  src.measurement_concept_id,
  src.measurement_date,
  src.measurement_datetime,
  src.measurement_time,
  src.measurement_type_concept_id,
  src.operator_concept_id,
  src.value_as_number,
  src.value_as_concept_id,
  src.unit_concept_id,
  src.range_low,
  src.range_high,
  src.provider_id,
  src.visit_occurrence_id,
  src.visit_detail_id,
  src.measurement_source_value,
  src.measurement_source_concept_id,
  src.unit_source_value,
  src.unit_source_concept_id,
  src.value_source_value,
  src.measurement_event_id,
  src.meas_event_field_concept_id
);

In [ ]:
%sql
SELECT * FROM _exponent.omop.measurement
LIMIT 20